In [4]:
import pandas as pd
import numpy as np

#  Charger le dataset
# On lit le fichier CSV contenant les données médicales
# Chaque ligne représente un patient
df = pd.read_csv("data.csv")

# Affichage des premières lignes pour vérifier
print("Aperçu du dataset :")
print(df.head())


# Nettoyage des données
# On supprime les colonnes inutiles pour l'apprentissage

# Supprimer la colonne vide si elle existe
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

# Supprimer la colonne id si elle existe
if 'id' in df.columns:
    df = df.drop(columns=['id'])

print("\nColonnes après nettoyage :")
print(df.columns)


# Transformer la variable cible
# La colonne diagnosis contient :
# M = Malignant
# B = Benign
#
# On transforme en valeurs numériques :
# M -> +1
# B -> -1
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': -1})


#  Séparer X et y
# X = variables explicatives
# y = variable cible

X = df.drop(columns=['diagnosis']).values.astype(float)
y = df['diagnosis'].values.astype(int)

print("\nDimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Aperçu du dataset :
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  ar

In [5]:
# Normalisation
# Important pour SVM :
# met toutes les variables entre 0 et 1

def normalize(X):
    X_min = X.min(axis=0)   # minimum de chaque colonne
    X_max = X.max(axis=0)   # maximum de chaque colonne
    
    # formule Min-Max
    return (X - X_min) / (X_max - X_min + 1e-8)

# Appliquer la normalisation
X = normalize(X)


#  Train / Test split
# On mélange les données pour éviter le biais
np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

# 80% pour apprentissage, 20% pour test
split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

# Création des datasets
X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]


In [6]:
#  SVM from scratch
# Implémentation d’un SVM linéaire
class SVMFromScratch:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, epochs=3000):
        # learning_rate : vitesse d’apprentissage
        # lambda_param : régularisation
        # epochs : nombre d’itérations
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.epochs = epochs
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialisation des poids et du biais
        self.w = np.zeros(n_features)
        self.b = 0.0

        # Boucle d’apprentissage
        for _ in range(self.epochs):
            for i in range(n_samples):
                x_i = X[i]
                y_i = y[i]

                # Vérifie si le point est bien classé
                condition = y_i * (np.dot(x_i, self.w) + self.b) >= 1

                if condition:
                    # Bien classé : seulement régularisation
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Mal classé : correction + régularisation
                    self.w -= self.lr * (2 * self.lambda_param * self.w - y_i * x_i)
                    self.b -= self.lr * (-y_i)

    def predict(self, X):
        # Calcul de la sortie linéaire
        linear_output = np.dot(X, self.w) + self.b

        # Fonction signe :
        # +1 ou -1
        return np.sign(linear_output)

In [7]:
#  Entraîner le modèle
model = SVMFromScratch()
model.fit(X_train, y_train)

#  Prédictions
# On applique le modèle sur les données de test
y_pred = model.predict(X_test)

In [8]:
#  Évaluation simple
# On compte les bonnes et mauvaises prédictions

correct = 0
incorrect = 0

for i in range(len(y_test)):
    if y_test[i] == y_pred[i]:
        correct += 1
    else:
        incorrect += 1

print("\nÉvaluation du modèle :")
print("Bonnes prédictions :", correct)
print("Mauvaises prédictions :", incorrect)
print("Total :", len(y_test))

#  Afficher quelques exemples
print("\nExemples de prédictions :")
for i in range(10):
    real = "Malignant" if y_test[i] == 1 else "Benign"
    pred = "Malignant" if y_pred[i] == 1 else "Benign"

    if y_test[i] == y_pred[i]:
        result = "Correct"
    else:
        result = "Faux"

    print(f"Réel: {real} | Prédit: {pred} --> {result}")

#  Paramètres appris
print("\nNombre de poids :", len(model.w))
print("Bias :", model.b)


Évaluation du modèle :
Bonnes prédictions : 107
Mauvaises prédictions : 7
Total : 114

Exemples de prédictions :
Réel: Malignant | Prédit: Malignant --> Correct
Réel: Malignant | Prédit: Malignant --> Correct
Réel: Benign | Prédit: Benign --> Correct
Réel: Malignant | Prédit: Malignant --> Correct
Réel: Malignant | Prédit: Malignant --> Correct
Réel: Benign | Prédit: Benign --> Correct
Réel: Benign | Prédit: Benign --> Correct
Réel: Benign | Prédit: Benign --> Correct
Réel: Malignant | Prédit: Malignant --> Correct
Réel: Benign | Prédit: Benign --> Correct

Nombre de poids : 30
Bias : -3.6089999999997135
